# 🧾 OCR hóa đơn TownHub — Train & Chạy (VietOCR + PaddleOCR)

Notebook chia **3 phần lớn**, chạy theo nhu cầu (dùng mục lục ▤ bên trái Colab để nhảy nhanh):

| Phần | Khi nào chạy |
|---|---|
| **A. Cấu hình** | ⭐ LUÔN chạy đầu tiên mỗi phiên |
| **B. Training** | Chỉ khi **chưa có** weight (A.2 sẽ báo). Train xong tự lưu Drive |
| **C. Chạy service** | Khi **đã có** weight trên Drive — có thể **bỏ qua B** |

> Đã train rồi và có weight trong Google Drive? → chạy **A** rồi nhảy thẳng **C**.


# ⚙️ PHẦN A — CẤU HÌNH  (luôn chạy đầu tiên)


### A.1 — Cài thư viện *(chạy 1 lần, sau đó Restart session)*
> Đám cảnh báo đỏ kiểu `jax/shap/... requires numpy>=2` là **vô hại** (các gói đó không dùng cho OCR), cứ bỏ qua.


In [ ]:
!nvidia-smi -L || echo '⚠️ Chưa bật GPU: Runtime → Change runtime type → T4 GPU'
!pip -q install fastapi uvicorn pydantic requests pdf2image pillow google-generativeai
!pip -q install torch easyocr vietocr paddlepaddle-gpu==2.6.1 paddleocr==2.7.3
# Ép NumPy 1.x cho khớp ABI của cv2/paddle (đây là bước xử lý lỗi 'numpy.core.multiarray failed to import').
!pip -q install 'numpy==1.26.4' 'opencv-python-headless==4.9.0.80'
print('✅ Cài xong. BÂY GIỜ: Runtime → Restart session, rồi chạy tiếp A.2 (KHÔNG chạy lại A.1).')


### A.2 — Thiết lập phiên *(chạy mỗi phiên / sau mỗi lần Restart)*
Clone code, mount Drive, trỏ đường dẫn weight. Cuối cell sẽ báo bạn cần **train** hay có thể **chạy thẳng**.


In [ ]:
# Chạy MỖI phiên (và mỗi lần sau khi Restart). Thiết lập thư mục + đường dẫn weight trên Drive.
import os
REPO_URL = 'https://github.com/thuongerikdev/TownHub'
if not os.path.exists('/content/townhub'):
    !git clone --depth 1 $REPO_URL /content/townhub
os.chdir('/content/townhub/ocr-service'); print('cwd:', os.getcwd())

from google.colab import drive; drive.mount('/content/drive')
DRIVE = '/content/drive/MyDrive/townhub_ocr'
os.makedirs(DRIVE, exist_ok=True)

# Service đọc weight qua các biến môi trường này (trỏ thẳng vào Drive).
os.environ['VIETOCR_WEIGHTS'] = f'{DRIVE}/weights/vietocr_invoice.pth'
os.environ['PADDLE_REC_DIR']  = f'{DRIVE}/inference/rec_vi'
os.environ['PADDLE_DET_DIR']  = f'{DRIVE}/inference/det_vi'
os.environ['PADDLE_REC_DICT'] = f'{DRIVE}/dict_vi.txt'
os.environ['GEMINIKEY'] = 'DAN_KEY_GEMINI_CUA_BAN'   # chỉ cần nếu dùng engine gemini
os.environ['OCRKEY']    = 'doan-ocr-2026'            # khớp OCR_API_KEY phía .NET

# Đã có weight trên Drive chưa? -> quyết định train hay chạy thẳng.
_have = all(os.path.exists(os.environ[k]) for k in ['VIETOCR_WEIGHTS','PADDLE_REC_DIR','PADDLE_DET_DIR'])
print('✅ ĐÃ có weight trên Drive → có thể BỎ QUA phần B, sang thẳng phần C (CHẠY).' if _have
      else 'ℹ️ CHƯA có weight → chạy phần B (TRAINING) trước.')


# 🏋️ PHẦN B — TRAINING  (bỏ qua nếu A.2 báo đã có weight)


### B.1 — Sinh dataset synthetic + lưu lên Drive


In [ ]:
!apt-get -qq install -y fonts-dejavu-core >/dev/null
!python training/make_dataset.py --n 800 --out ./dataset --fonts /usr/share/fonts/truetype/dejavu
# Lưu dataset lên Drive dưới dạng 1 file .zip.
# (Dataset có ~42k ảnh nhỏ — copy TỪNG file lên Drive mount rất chậm, có thể treo hàng giờ;
#  nén 1 file rồi copy chỉ mất vài giây.)
!zip -rq dataset.zip dataset && cp dataset.zip {DRIVE}/dataset.zip \
 && echo '✅ Đã lưu dataset.zip → Drive/townhub_ocr/'


> Muốn **dùng lại dataset cũ** trên Drive thay vì sinh mới (nhanh hơn):
> `!cp {DRIVE}/dataset.zip . && unzip -q -o dataset.zip`


In [ ]:
# (tuỳ chọn) Xem thử 1 hóa đơn + vài crop
from PIL import Image; import glob
display(Image.open('dataset/det/images/inv_00001.jpg'))
for f in sorted(glob.glob('dataset/rec/images/inv_00001_*.jpg'))[:5]: display(Image.open(f))


### B.2 — Fine-tune VietOCR (recognition) + lưu Drive


In [ ]:
!python training/finetune_vietocr.py --data ./dataset/rec --iters 15000 --batch 32 \
        --out ./weights/vietocr_invoice.pth
!mkdir -p {DRIVE}/weights && cp weights/vietocr_invoice.pth {DRIVE}/weights/ \
 && echo '✅ Đã lưu VietOCR → Drive'


### B.3 — Fine-tune PaddleOCR (detection + recognition) + lưu Drive


In [ ]:
%cd /content/townhub/ocr-service
# Clone đúng nhánh release/2.7 để KHỚP với paddleocr==2.7.3 (nhánh mặc định là 3.x, khác cấu trúc config).
!test -d PaddleOCR || git clone --depth 1 -b release/2.7 https://github.com/PaddlePaddle/PaddleOCR.git


**B.3.1 — Tải model pretrain** (dùng `wget -c`, KHÔNG dùng `-q`; cuối cell `find` phải ra: rec `student.pdparams`, det `best_accuracy.pdparams`)


In [ ]:
import os; os.chdir('/content/townhub/ocr-service/PaddleOCR'); print('cwd:', os.getcwd())
!mkdir -p pretrain
!cd pretrain && wget -c https://paddleocr.bj.bcebos.com/PP-OCRv4/chinese/ch_PP-OCRv4_rec_train.tar && tar xf ch_PP-OCRv4_rec_train.tar
!cd pretrain && wget -c https://paddleocr.bj.bcebos.com/PP-OCRv4/chinese/ch_PP-OCRv4_det_train.tar && tar xf ch_PP-OCRv4_det_train.tar
print('=== KIỂM TRA (phải thấy 2 file .pdparams; nếu trống là tải lỗi) ===')
!find pretrain -name '*.pdparams'


**B.3.2 — Train RECOGNITION** (tự động: lần đầu → pretrained, đã có → train tiếp; lưu Drive mỗi epoch. OOM thì hạ 64→32)


In [ ]:
import os
from google.colab import drive; drive.mount('/content/drive')
os.chdir('/content/townhub/ocr-service/PaddleOCR')
SAVE = '/content/drive/MyDrive/townhub_ocr/output_rec_vi'   # lưu checkpoint thẳng lên Drive
os.makedirs(SAVE, exist_ok=True)
if os.path.exists(f'{SAVE}/latest.pdparams'):
    init = f'Global.checkpoints={SAVE}/latest'; print('▶ Có checkpoint → TRAIN TIẾP từ latest')
else:
    init = 'Global.pretrained_model=./pretrain/ch_PP-OCRv4_rec_train/student'; print('▶ Lần đầu → từ pretrained')

!python tools/train.py -c configs/rec/PP-OCRv4/ch_PP-OCRv4_rec.yml \
  -o {init} \
     Global.character_dict_path=../dataset/dict_vi.txt \
     Global.use_space_char=True Global.epoch_num=80 \
     Global.save_model_dir={SAVE} \
     Train.loader.batch_size_per_card=64 Train.sampler.first_bs=64 \
     Eval.loader.batch_size_per_card=64 \
     Train.dataset.data_dir=../dataset/rec \
     Train.dataset.label_file_list=['../dataset/rec/train.txt'] \
     Eval.dataset.data_dir=../dataset/rec \
     Eval.dataset.label_file_list=['../dataset/rec/val.txt']


**B.3.3 — Train DETECTION** (tự động resume + lưu Drive; ảnh cả trang nên batch 8, OOM thì hạ 8→4)


In [ ]:
import os
from google.colab import drive; drive.mount('/content/drive')
os.chdir('/content/townhub/ocr-service/PaddleOCR')
SAVE = '/content/drive/MyDrive/townhub_ocr/output_det_vi'   # thư mục Drive RIÊNG cho det
os.makedirs(SAVE, exist_ok=True)
if os.path.exists(f'{SAVE}/latest.pdparams'):
    init = f'Global.checkpoints={SAVE}/latest'; print('▶ Có checkpoint → TRAIN TIẾP từ latest')
else:
    init = 'Global.pretrained_model=./pretrain/ch_PP-OCRv4_det_train/best_accuracy'; print('▶ Lần đầu → từ pretrained')

!python tools/train.py -c configs/det/ch_PP-OCRv4/ch_PP-OCRv4_det_student.yml \
  -o {init} \
     Global.epoch_num=150 \
     Global.save_model_dir={SAVE} \
     Train.loader.batch_size_per_card=8 Eval.loader.batch_size_per_card=8 \
     Train.dataset.data_dir=../dataset/det \
     Train.dataset.label_file_list=['../dataset/det/train_label.txt'] \
     Eval.dataset.data_dir=../dataset/det \
     Eval.dataset.label_file_list=['../dataset/det/val_label.txt']


**B.3.4 — Export inference model + lưu Drive** (đọc best_accuracy từ Drive)


In [ ]:
import os
DRIVE = '/content/drive/MyDrive/townhub_ocr'

# --- Sau khi Restart runtime, đĩa local bị xoá: clone lại repo + PaddleOCR nếu thiếu ---
if not os.path.exists('/content/townhub'):
    !git clone --depth 1 https://github.com/thuongerikdev/TownHub /content/townhub
if not os.path.exists('/content/townhub/ocr-service/PaddleOCR'):
    !cd /content/townhub/ocr-service && git clone --depth 1 -b release/2.7 https://github.com/PaddlePaddle/PaddleOCR.git

# --- Lấy dict_vi.txt. Trên Drive chỉ lưu dataset.zip nên dict nằm TRONG zip đó
#     -> rút đúng 1 file 'dataset/dict_vi.txt' ra (không cần bung cả 42k ảnh), rồi lưu ra Drive để service dùng lại.
DICT = f'{DRIVE}/dict_vi.txt'
if not os.path.exists(DICT):
    if os.path.exists('/content/townhub/ocr-service/dataset/dict_vi.txt'):        # vừa train xong trong phiên này
        !cp /content/townhub/ocr-service/dataset/dict_vi.txt {DICT}
    elif os.path.exists(f'{DRIVE}/dataset.zip'):                                   # phiên mới: rút từ zip trên Drive
        !unzip -q -o {DRIVE}/dataset.zip 'dataset/dict_vi.txt' -d /content/townhub/ocr-service
        !cp /content/townhub/ocr-service/dataset/dict_vi.txt {DICT}
assert os.path.exists(DICT), '❌ Không có dict_vi.txt lẫn dataset.zip trên Drive — cần chạy lại B.1 để sinh dataset.'
print('dict:', DICT, '-> OK')

# --- Chọn checkpoint TỪ DRIVE: ưu tiên best_accuracy, chưa có thì dùng latest (train dở vẫn export được) ---
def pick(dirp):
    if os.path.exists(f'{dirp}/best_accuracy.pdparams'): return f'{dirp}/best_accuracy'
    if os.path.exists(f'{dirp}/latest.pdparams'):        return f'{dirp}/latest'
    raise FileNotFoundError(f'❌ Không thấy checkpoint (best_accuracy/latest) trong {dirp} trên Drive.')
REC_CKPT = pick(f'{DRIVE}/output_rec_vi')
DET_CKPT = pick(f'{DRIVE}/output_det_vi')
print('rec checkpoint:', REC_CKPT, '| det checkpoint:', DET_CKPT)

os.chdir('/content/townhub/ocr-service/PaddleOCR')
# REC: BẮT BUỘC truyền character_dict_path + use_space_char GIỐNG HỆT lúc train,
#      nếu không lớp output theo dict tiếng Trung sẽ lệch shape với best_accuracy → lỗi nạp weight.
!python tools/export_model.py -c configs/rec/PP-OCRv4/ch_PP-OCRv4_rec.yml \
  -o Global.pretrained_model={REC_CKPT} \
     Global.character_dict_path={DICT} Global.use_space_char=True \
     Global.save_inference_dir=./inference/rec_vi
# DET: không dùng bảng ký tự, chỉ cần checkpoint từ Drive.
!python tools/export_model.py -c configs/det/ch_PP-OCRv4/ch_PP-OCRv4_det_student.yml \
  -o Global.pretrained_model={DET_CKPT} \
     Global.save_inference_dir=./inference/det_vi

%cd /content/townhub/ocr-service
!mkdir -p {DRIVE}/inference && cp -r PaddleOCR/inference/rec_vi PaddleOCR/inference/det_vi {DRIVE}/inference/ \
 && echo '✅ inference (rec_vi, det_vi) + dict → Drive'


# 🚀 PHẦN C — CHẠY SERVICE  (khi đã có weight; có thể bỏ qua B)


### C.1 — Kiểm tra weight


In [ ]:
# Kiểm tra weight đã sẵn trên Drive chưa trước khi chạy service.
import os
for k in ['VIETOCR_WEIGHTS','PADDLE_REC_DIR','PADDLE_DET_DIR','PADDLE_REC_DICT']:
    ok = os.path.exists(os.environ[k])
    print(('✅' if ok else '❌ THIẾU'), k, '=', os.environ[k])
print('\nCó ❌ nghĩa là chưa train (chạy phần B) hoặc chưa lưu lên Drive.')


### C.2 — Mở tunnel cloudflared


In [ ]:
# Mở tunnel cloudflared -> URL https công khai để backend .NET gọi vào (đặt vào OCR_SERVICE_URL).
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /usr/local/bin/cloudflared && chmod +x /usr/local/bin/cloudflared
import subprocess, re
p = subprocess.Popen(['cloudflared','tunnel','--url','http://localhost:7860','--no-autoupdate'],
                     stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
for line in p.stdout:
    print(line, end='')
    m = re.search(r'https://[-a-z0-9]+\.trycloudflare\.com', line)
    if m: print('\n🌐 OCR_SERVICE_URL =', m.group(0)); break


### C.3 — Chạy service *(giữ cell này chạy)*
Đặt `OCR_SERVICE_URL` phía backend .NET = URL cloudflared ở C.2, `OCR_API_KEY` = `OCRKEY`.


In [ ]:
!python app.py


# 📊 PHẦN D — PHÂN TÍCH & SO SÁNH (cho báo cáo)

Chạy **sau khi đã fine-tune xong** (rec + det + export + lưu Drive). So sánh **3 engine** trên cùng tập val — **EasyOCR ↔ VietOCR ↔ PaddleOCR** — kèm **ensemble (kết hợp)**, sinh **17 biểu đồ** + `metrics_summary.csv` vào `./analysis` rồi copy lên Drive:

- **Huấn luyện:** đường cong Loss/Accuracy, lịch learning-rate, TRƯỚC vs SAU fine-tune.
- **So sánh engine:** cột Seq/Char/NED, radar 5 trục, độ trễ & thông lượng, đánh đổi chính xác↔tốc độ.
- **Phân tích lỗi:** histogram/boxplot/CDF của CER, độ chính xác theo độ dài chuỗi, cơ cấu lỗi (sub/ins/del), top ký tự đọc sai, cặp nhầm lẫn.
- **Kết hợp:** giao tập đọc đúng (kiểu Venn), lợi ích ensemble (vote vs oracle chặn trên), bảng tổng hợp.

> EasyOCR đã cài ở A.1; lần chạy đầu nó tự tải model tiếng Việt (~vài chục MB). Muốn bỏ EasyOCR thì thêm cờ `--no-easyocr` vào lệnh D.1.


### D.1 — Chạy phân tích (đọc log + đánh giá trên tập val)


In [ ]:
!pip -q install matplotlib
import os; os.chdir('/content/townhub/ocr-service')
D = '/content/drive/MyDrive/townhub_ocr'
# Cần ảnh val local để đánh giá; phiên mới chưa có thì bung từ dataset.zip trên Drive.
if not os.path.exists('./dataset/rec/val.txt'):
    !cp {D}/dataset.zip . && unzip -q -o dataset.zip && echo '📦 đã bung dataset từ Drive'
!python training/analyze.py \
  --data ./dataset/rec --label ./dataset/rec/val.txt \
  --viet {D}/weights/vietocr_invoice.pth \
  --rec  {D}/inference/rec_vi \
  --dict {D}/dict_vi.txt \
  --log  {D}/output_rec_vi/train.log \
  --limit 500


### D.2 — Hiện biểu đồ + lưu Drive


In [ ]:
from IPython.display import Image, display
import glob
for f in sorted(glob.glob('analysis/*.png')):
    print(f); display(Image(f))
!cp -r analysis /content/drive/MyDrive/townhub_ocr/analysis && echo '✅ đã lưu analysis → Drive'
